# RAG local — Qwen3.5 9B / Ollama + Gold + MLflow

Execute localmente depois de instalar o projeto e copiar `.env.example` para `.env`.
Configure o SQL Warehouse e autentique com `databricks auth login --profile projeto`.
O modelo `qwen3.5:9b` gera respostas; `embeddinggemma:latest` é usado somente
quando `backend=dense`. A Gold já está segmentada: não há nova divisão dos trechos.

In [ ]:
from pathlib import Path
import sys
from dataclasses import replace
from dotenv import load_dotenv

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src/rag_cognus").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")
from rag_cognus.config import Settings
from rag_cognus.gold import load_from_databricks, load_snapshot, save_snapshot
from rag_cognus.retrieval import GoldRetriever
from rag_cognus.chain import RagPipeline
from rag_cognus.evaluation import configure_tracking, evaluate, load_cases

settings = Settings.from_env()
SNAPSHOT = PROJECT_ROOT / ".cache/gold.jsonl"
# Atualização explícita: nenhuma troca silenciosa de snapshot durante os experimentos.
ATUALIZAR_SNAPSHOT = False
if ATUALIZAR_SNAPSHOT or not SNAPSHOT.exists():
    documents = load_from_databricks(settings)
    print(save_snapshot(documents, SNAPSHOT, settings.gold_view))
documents = load_snapshot(SNAPSHOT)
configure_tracking(settings)
print("Gold:", len(documents), "trechos; modelo:", settings.chat_model)

In [ ]:
# Troque para "dense" para comparar com embeddings locais.
config = replace(settings, backend="bm25", top_k=5)
retriever = GoldRetriever(documents, config)
rag = RagPipeline(retriever, config)
answer = rag.predict("Como organizar uma apresentação clara para meu público?")
print(answer["response"])
print(answer["citations"])

In [ ]:
cases = load_cases(PROJECT_ROOT / "data/evaluation/questions.jsonl")
# Comece pequeno: cada pergunta usa uma chamada ao Qwen se houver contexto.
summary = evaluate(cases[:2], retriever, config, pipeline=rag,
                   output_dir=PROJECT_ROOT / "outputs", run_name="qwen-smoke")
print(summary)
if summary["metrics"]["error_rate"]:
    raise RuntimeError("Consulte o arquivo per_case.json do run.")

## Avaliação por juiz opcional

`evaluate_groundedness` usa o próprio Qwen como juiz e registra resultados com
`mlflow.genai.evaluate`. É uma autoavaliação exploratória, sujeita ao viés do mesmo
modelo; não equivale a revisão humana ou avaliação clínica. Abstenções recebem
`not_scored`, não aprovação automática de groundedness.

Para usar outro juiz, adapte a configuração do modelo de forma explícita.
Inicie o servidor com `rag-cognus mlflow-local` e abra `http://127.0.0.1:5000`.
Para executar a suite completa, use `python scripts/eval_local.py --judge`.

In [ ]:
# Ative somente quando quiser fazer chamadas adicionais ao modelo local.
EXECUTAR_JUIZ = False
if EXECUTAR_JUIZ:
    from rag_cognus.evaluation import evaluate_groundedness
    results = PROJECT_ROOT / "outputs" / f"{summary['run_id']}.json"
    print(evaluate_groundedness(results, settings))